## LAB 02 — Entendendo os dados com o Assistant (Social)

1. Rode o **LAB 01** antes (tabela Delta no mesmo `catalog_name` / `schema_name`).
2. Confira os widgets.
3. Cluster ou serverless → **Run All**.
4. Use o Assistant (`Ctrl + I`) só com as 27 colunas do CSV.
5. Não pergunte sobre faturas, CNPJ, `engagement_rate` nativo ou custo de patrocínio — **não existem** neste CSV.

In [ ]:
def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "social")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")
print(f"{catalog_name}.{schema_name}")

In [ ]:
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}"))

### Passo 1 — Perguntas abertas (Databricks Assistant)

1. Copie uma pergunta.
2. Clique no Assistant ao lado da célula SQL ou use `Ctrl + I`.
3. Peça SQL que use **somente** as 27 colunas de `social_media_dataset`.
4. Execute o código sugerido.

Engagement rate **derivado** = `(likes + shares + comments_count) / views`. No dump isso quase não se move (≈ 0,199). Comparar orgânico vs patrocinado **sem** controlar `platform`, `content_category` e faixa de `follower_count` é média crua — fora do brief.

## Perguntas (colunas reais do CSV)

- Quantos posts por `platform`, `content_type` e `content_category`? Inclua Bilibili e RedNote.
- Taxa derivada `(likes + shares + comments_count) / views` por plataforma e por `is_sponsored` — a média crua muda?
- A mesma taxa **controlada**: `platform` × `content_category` × `is_sponsored`.
- `creator_id` vs `creator_name`: quantos nomes distintos por creator?
- `audience_age_distribution` / `audience_gender_distribution` / `audience_location` — isso é JSON ou um rótulo?
- Posts com `views` no fundo da faixa vs no topo — a taxa muda ou o dump é quase constante?
- `hashtags` nulos e tokens (vírgula, sem `#`). Dá para combinar sem saneamento?
- O que **não** funciona: menor taxa controlada (plataforma × tipo × categoria × sponsor).

In [ ]:
display(
    spark.sql(
        """
SELECT
  COUNT(*) AS posts,
  COUNT(DISTINCT platform) AS platforms,
  COUNT(DISTINCT creator_id) AS creators,
  COUNT(DISTINCT creator_name) AS creator_names,
  SUM(CASE WHEN is_sponsored THEN 1 ELSE 0 END) AS sponsored,
  SUM(CASE WHEN NOT is_sponsored THEN 1 ELSE 0 END) AS organic,
  ROUND(AVG((likes + shares + comments_count) / views), 4) AS avg_engagement_per_view
FROM social_media_dataset
"""
    )
)

In [ ]:
display(
    spark.sql(
        """
SELECT
  platform,
  content_category,
  is_sponsored,
  COUNT(*) AS posts,
  ROUND(AVG((likes + shares + comments_count) / views), 4) AS avg_engagement_per_view,
  ROUND(AVG(follower_count), 0) AS avg_followers
FROM social_media_dataset
GROUP BY platform, content_category, is_sponsored
ORDER BY platform, content_category, is_sponsored
"""
    )
)

In [ ]:
display(
    spark.sql(
        """
SELECT
  creator_id,
  COUNT(DISTINCT creator_name) AS names,
  COUNT(*) AS posts,
  MIN(follower_count) AS min_followers,
  MAX(follower_count) AS max_followers
FROM social_media_dataset
GROUP BY creator_id
HAVING COUNT(DISTINCT creator_name) > 1
ORDER BY names DESC
LIMIT 10
"""
    )
)

### Bônus — prompt para o Assistant

> Estude as 27 colunas de `social_media_dataset`. Não invente `engagement_rate` nem custo. Derive `(likes+shares+comments_count)/views` e diga se a média crua orgânico vs patrocinado se move. Controle por `platform`, `content_category` e faixa de `follower_count`. Use `creator_id`, não `creator_name`. Inclua Bilibili e RedNote. Liste o que parar de fazer e um threshold de sponsor só se os dados sustentarem. Não exponha `content_url` em dump.

### Genie

Sala publicada: [lab-004-social-genie](https://dbc-0dea39b3-be1e.cloud.databricks.com/genie/rooms/01f1b6165d6e1a8bbb831d2a78fd6929?o=7474644277755368) (`01f1b6165d6e1a8bbb831d2a78fd6929`).

O chatbot local (`e2e-chatbot-app-next`) abre esta sala em `/room/social`.

Sem engagement_rate nativo. Use creator_id.
